# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import pandas as pd
import torch
import os
import numpy as np
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

In [3]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


In [ ]:
# Commented to avoid accidental deletion

# import shutil
# import os

# # Delete old folder
# if os.path.exists("./dataset_processed"):
#     shutil.rmtree("./dataset_processed")
#     print("Folder deleted. Ready to regenerate.")

Folder deleted. Ready to regenerate.


# Configuration

## Set values

In [15]:
# --- CONFIGURATION ---
CSV_PATH = "/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv"
BASE_OUTPUT_DIR = "./dataset_processed_thu0103"
TIME_WINDOW_SEC = 30  # Each graph represents 30 seconds
CHUNK_SIZE = 1000000   # Rows to read at a time in RAM


In [16]:
!# Create directories
os.makedirs(os.path.join(BASE_OUTPUT_DIR, 'test2'), exist_ok=True)


In [6]:
cicids2018v3_thu0103_orig = pd.read_csv(CSV_PATH)
cicids2018v3_thu0103 = cicids2018v3_thu0103_orig.copy()
# Convert 'FLOW_START_TIME' to datetime type after loading
cicids2018v3_thu0103['FLOW_START_TIME'] = pd.to_datetime(cicids2018v3_thu0103['FLOW_START_TIME'])

timestamps = cicids2018v3_thu0103['FLOW_START_TIME']
GLOBAL_START_TIME = timestamps.min()
GLOBAL_END_TIME = timestamps.max()
TOTAL_DURATION = GLOBAL_END_TIME - GLOBAL_START_TIME

print(f"Global start time: {GLOBAL_START_TIME}")
print(f"Global end time: {GLOBAL_END_TIME}")


Global start time: 2018-03-01 00:00:03.289000
Global end time: 2018-03-01 23:30:54.233000


## Select features

In [7]:
NUMERIC_FEATURES = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG',
    'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_STDDEV',
    'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
    'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_PKTS',
    'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
    'TCP_FLAGS', 'MIN_TTL', 'MAX_TTL'
]

CATEGORICAL_FEATURES = [
    'PROTOCOL', 'L4_DST_PORT'
]

# Define functions

## Protocol and Port One-Hot

In [8]:
# One-Hot for Protocol and for Port

def get_protocol_vector(proto):
    """
    One-Hot for Protocol (5 dimensions):
    [TCP(6), UDP(17), ICMP(1 or 58 for ICMPv6), IGMP(2), Other]
    """
    if proto == 6:   return [1, 0, 0, 0, 0] # TCP
    if proto == 17:  return [0, 1, 0, 0, 0] # UDP
    if proto in (1, 58):   return [0, 0, 1, 0, 0] # ICMP or ICMPv6
    if proto == 2:  return [0, 0, 0, 1, 0] # IGMP
    return [0, 0, 0, 0, 1] # Other

def get_port_role_vector(port):
    """
    Classifies the destination port using a 7-dimensional One-Hot vector.
    Optimized for the NF-CSE-CIC-IDS2018-v3 dataset and infiltration attacks.

    [0] Web (HTTP/S, Proxies, RPC-JSON)
    [1] Admin/Remote (SSH, RDP, Telnet, VNC, ADB) - Incluye variantes
    [2] Windows/SMB (NetBIOS, RPC, SMB) - Clave para Mov. Lateral
    [3] DNS/Infra (DNS, LLMNR, DHCP, NTP, SSDP, SIP)
    [4] Database (SQL)
    [5] Other Privileged (< 1024) - Incluye Port 0
    [6] Other High/Ephemeral (>= 1024)
    """

    # 1. Web & HTTP-like
    # 80/443 (Std), 8080/81/3128 (Proxies/Alt), 8545 (Ethereum/RPC)
    if port in [80, 443, 8080, 8443, 81, 3128, 8545]:
        return [1, 0, 0, 0, 0, 0, 0]

    # 2. Admin & Remote Access
    # 22/222/2222 (SSH), 23/2323 (Telnet), 3389/3390/3394 (RDP),
    # 5900/5901 (VNC), 5555 (ADB - Android), 21/2131 (FTP)
    elif port in [22, 222, 2222, 23, 2323, 3389, 3390, 3394, 5900, 5901, 5555, 21, 2131]:
        return [0, 1, 0, 0, 0, 0, 0]

    # 3. Windows Services / SMB
    # 445 (SMB), 135 (RPC Endpoint), 137/138/139 (NetBIOS)
    elif port in [445, 135, 137, 138, 139]:
        return [0, 0, 1, 0, 0, 0, 0]

    # 4. DNS & Infrastructure
    # 53 (DNS), 5355 (LLMNR), 67/547 (DHCP), 123 (NTP), 1900 (SSDP/UPnP), 5060 (SIP)
    elif port in [53, 5355, 67, 547, 123, 1900, 5060]:
        return [0, 0, 0, 1, 0, 0, 0]

    # 5. Database
    # 1433 (MSSQL), 3306 (MySQL)
    elif port in [1433, 3306, 5432, 6379, 27017]:
        return [0, 0, 0, 0, 1, 0, 0]

    # 6. Other Privileged Users (< 1024)
    # Port 0 and other legacy services will be taken offline here
    elif port < 1024:
        return [0, 0, 0, 0, 0, 1, 0]

    # 7. High Ports / Ephemeral (>= 1024)
    else:
        return [0, 0, 0, 0, 0, 0, 1]

In [9]:
edge_attr_dim = len(NUMERIC_FEATURES) + 5 + 7 # one-hot protocol and one-hot port
print(f"edge_attr dimension: {edge_attr_dim}")

edge_attr dimension: 32


## Get an ID for each IP

In [10]:
# Map IP->ID

ip_map = {}
next_id = 0

def get_ip_id(ip_str):
    global next_id
    if ip_str not in ip_map:
        ip_map[ip_str] = next_id
        next_id += 1
    return ip_map[ip_str]


## Create empty graphs

In [11]:
# Auxiliar function for save empty graphs for time windows without nodes

def save_empty_graph(window_id):
    # Determine which split it belongs to based on the window START time
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)
    split = 'test2'

    # Create empty graph
    # Empty tensors with the correct shape (0 rows, N columns)
    x = torch.empty((0, 16), dtype=torch.float)
    edge_index = torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.empty((0, edge_attr_dim), dtype=torch.float)
    y = torch.empty((0), dtype=torch.float) # No labels

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.global_node_ids = torch.empty((0), dtype=torch.long)
    data.timestamp = window_start_time
    data.is_empty = True # Flag for model

    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))


## Create graphs (not empty)

In [12]:
def save_window_group(window_id, df_group):
    # Determine which split it belongs to based on the window START time
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)
    split = 'test2'

    # Graph nodes: GLOBAL IDS
    src_global = [get_ip_id(ip) for ip in df_group['IPV4_SRC_ADDR']]
    dst_global = [get_ip_id(ip) for ip in df_group['IPV4_DST_ADDR']]

    # MAP: GLOBAL -> LOCAL
    ## Unique nodes that are in THIS window
    unique_global_nodes = sorted(list(set(src_global + dst_global)))

    ## Dictionary: {Global_ID: Local_ID (0, 1, 2...)}
    global_to_local = { gid: lid for lid, gid in enumerate(unique_global_nodes) }

    # RE-INDEX EDGES (Using Local IDs)
    src_local = [global_to_local[gid] for gid in src_global]
    dst_local = [global_to_local[gid] for gid in dst_global]

    edge_index = torch.tensor([src_local, dst_local], dtype=torch.long)

    # Features and normalization
    ## 1. Numeric Features (Log1p + StandardScaler in main process loop)
    numeric_tensor = torch.tensor(df_group[NUMERIC_FEATURES].values.astype('float32'))

    ## 2. Dst Port (One-Hot)
    port_list = df_group['L4_DST_PORT'].apply(get_port_role_vector).tolist()
    port_tensor = torch.tensor(port_list, dtype=torch.float32)

    ## 3. Protocol (One-Hot)
    proto_list = df_group['PROTOCOL'].apply(get_protocol_vector).tolist()
    proto_tensor = torch.tensor(proto_list, dtype=torch.float32)

    ## 4. Concatenate everything to form the final edge_attr
    edge_attr = torch.cat([port_tensor, proto_tensor, numeric_tensor], dim=1)

    # Nodes (Identity Agnostic)
    num_nodes = len(unique_global_nodes)
    x = torch.ones((num_nodes, 16), dtype=torch.float)

    # Target
    y = torch.tensor(
        df_group['Attack'].apply(lambda x: 1 if x == 'Infilteration' else 0).values,
        dtype=torch.float
    )

    # Put together
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

    # IMPORTANT: We keep a record of the actual global IDs
    # This is what the ST-GNN will use to retrieve the correct memory
    data.global_node_ids = torch.tensor(unique_global_nodes, dtype=torch.long)
    data.timestamp = window_start_time

    # Save: We use the window_id as the name to maintain strict order
    # graph_00000.pt, graph_00001.pt ...
    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))

# Complete process loop

In [13]:
import joblib

print("STEP 1: Loading the Scaler of the day wednesday 28/02/2018...")

scaler = joblib.load('/content/drive/MyDrive/nids-mitre/dataset_processed/scaler_wed2802.pkl')
print("Scaler loaded.")

STEP 1: Loading the Scaler of the day wednesday 28/02/2018...
Scaler loaded.


In [18]:
# PROCESS LOOP

print("STEP 2: Generating Graphs...")

expected_window_id = 0 # Counter for tracking continuity
buffer_df = pd.DataFrame()
print("Processing chunks...")

# Iterador de chunks
chunk_iterator = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)

for chunk_idx, chunk in enumerate(chunk_iterator):
    mask_valid_ips = (chunk['IPV4_SRC_ADDR'] != '0.0.0.0') & (chunk['IPV4_DST_ADDR'] != '0.0.0.0')
    chunk = chunk[mask_valid_ips]
    chunk = chunk.copy()
    chunk['FLOW_START_TIME'] = pd.to_datetime(chunk['FLOW_START_TIME'])

    # Apply Log1p and Scaler
    # Note: It doesn't matter if it's Train, Val or Test, they all transform with the scaler already trained
    if not chunk.empty:
        vals = chunk[NUMERIC_FEATURES].values
        vals = np.log1p(vals)
        vals = scaler.transform(vals)

        # We overwrite the columns in the chunk's dataframe
        chunk[NUMERIC_FEATURES] = vals

    # 1. Join with previous buffer (if one exists)
    df = pd.concat([buffer_df, chunk])

    # 2. Assign Window ID to each row (Vectorized = Very Fast)
    # Subtract the global starting value and divide by 30. The floor (int) gives us the ID.
    df['window_id'] = ((df['FLOW_START_TIME'] - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC)).astype(int)

    # 3. Identify which windows we have in this chunk
    present_windows = df['window_id'].unique()
    present_windows.sort()

    if len(present_windows) > 0:
        # 4. SECURITY STRATEGY:
        # The last window in this chunk may be incomplete.
        # (Your data may continue in the next chunk.)
        # Therefore, we process all windows EXCEPT the last one.
        last_window_id = present_windows[-1]

        # Separate: Complete vs Incomplete
        df_complete = df[df['window_id'] < last_window_id]
        df_incomplete = df[df['window_id'] == last_window_id]

        # 5. Process entire groups
        for win_id, group in df_complete.groupby('window_id'):
            # 1. Fill in the gaps if we skip numbers
            while expected_window_id < win_id:
                save_empty_graph(expected_window_id)
                expected_window_id += 1

            # 2. Save the current window (which does contain data)
            save_window_group(win_id, group)
            expected_window_id += 1

        # 6. Save the incomplete part in the buffer for the next chunk
        buffer_df = df_incomplete
    else:
        # Rare case: The entire chunk belongs to a window that was already in the buffer
        buffer_df = df

    print(f"Chunk {chunk_idx} processed. Buffer size: {len(buffer_df)}")


# FINAL CLEANING
# At the end, what remains in the buffer is the last valid window
if not buffer_df.empty:
    print("Processing last buffer...")
    for win_id, group in buffer_df.groupby('window_id'):
        # Fill in any gaps before the last window, if there are any
        while expected_window_id < win_id:
            save_empty_graph(expected_window_id)
            expected_window_id += 1
        save_window_group(win_id, group)
        expected_window_id += 1

# Fill any gaps after the last processed graph until the end of the dataset
# Let's calculate the ID of the last possible window.
final_possible_window_id = ((GLOBAL_END_TIME - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC))
while expected_window_id <= final_possible_window_id:
    save_empty_graph(expected_window_id)
    expected_window_id += 1

print("Dataset generation complete!")

STEP 2: Generating Graphs...
Processing chunks...
Chunk 0 processed. Buffer size: 750
Chunk 1 processed. Buffer size: 179
Chunk 2 processed. Buffer size: 10
Processing last buffer...
Dataset generation complete!
